# LLM Decoding Strategies
### Following: *Build a Large Language Model From Scratch* — Sebastian Raschka

---

In Chapter 5, after we built the GPT-2 architecture and saw that untrained models produce gibberish, we now focus on **how** we sample tokens from the model's output distribution.

We will cover:
1. **Greedy Decoding** — always pick the highest probability token
2. **Temperature Scaling** — control the sharpness of the distribution
3. **Top-k Sampling** — restrict sampling to the top-k most likely tokens
4. **Nucleus (Top-p) Sampling** — restrict sampling to a cumulative probability mass
5. **Full `generate()` Function** — combining temperature + top-k for practical use


## Step 1: Imports & GPT-2 Configuration

In [ ]:
import torch
import torch.nn as nn
import tiktoken
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size (BPE tokenizer)
    "context_length": 1024,  # Maximum context / sequence length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "n_layers": 12,          # Number of transformer blocks
    "drop_rate": 0.1,        # Dropout probability
    "qkv_bias": False        # Bias in Q, K, V projections
}

print("GPT-2 124M Config:")
for k, v in GPT_CONFIG_124M.items():
    print(f"  {k}: {v}")

---
## Step 2: Full GPT-2 Architecture (from scratch)

We re-define every building block so this notebook is fully self-contained.

### 2.1 Layer Normalization

GPT-2 uses **Pre-LayerNorm** — normalisation is applied *before* the attention and feed-forward sub-layers (unlike the original Transformer which normalises *after*).

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

In [ ]:
class LayerNorm(nn.Module):
    """Layer Normalization with learnable scale (gamma) and shift (beta)."""

    def __init__(self, emb_dim):
        super().__init__()
        self.eps   = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))   # gamma
        self.shift = nn.Parameter(torch.zeros(emb_dim))  # beta

    def forward(self, x):
        mean   = x.mean(dim=-1, keepdim=True)
        var    = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

### 2.2 GELU Activation

GPT-2 uses **GELU** instead of ReLU. The approximation used in the original paper is:

$$\text{GELU}(x) = 0.5 \cdot x \cdot \left(1 + \tanh\!\left[\sqrt{\frac{2}{\pi}}\left(x + 0.044715\, x^3\right)\right]\right)$$

In [ ]:
class GELU(nn.Module):
    """Gaussian Error Linear Unit — approximation used in GPT-2."""

    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


# Quick visualisation
x = torch.linspace(-3, 3, 200)
gelu = GELU()
plt.figure(figsize=(6, 3))
plt.plot(x.numpy(), gelu(x).numpy(), label="GELU", color="steelblue")
plt.plot(x.numpy(), torch.relu(x).numpy(), label="ReLU", color="tomato", linestyle="--")
plt.title("GELU vs ReLU")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.3 Feed-Forward Network

Each Transformer block contains a two-layer MLP that expands the dimension by 4× then contracts it back:

$$\text{FFN}(x) = W_2 \cdot \text{GELU}(W_1 x + b_1) + b_2$$

where $W_1 \in \mathbb{R}^{d \times 4d}$ and $W_2 \in \mathbb{R}^{4d \times d}$.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network (expand 4x, contract 1x)."""

    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),  # expand
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),  # contract
        )

    def forward(self, x):
        return self.layers(x)

### 2.4 Multi-Head Causal Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$$

where $M$ is the **causal mask** that sets future positions to $-\infty$ before the softmax.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Causal (masked) Self-Attention."""

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out    = d_out
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads

        self.W_query  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key    = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value  = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)          # combine heads
        self.dropout  = nn.Dropout(dropout)

        # Upper-triangular causal mask (True = masked out)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        Q = self.W_query(x)   # (b, T, d_out)
        K = self.W_key(x)
        V = self.W_value(x)

        # Split into heads: (b, T, d_out) -> (b, num_heads, T, head_dim)
        Q = Q.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention scores
        attn_scores = Q @ K.transpose(2, 3)  # (b, heads, T, T)

        # Apply causal mask
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Weighted sum: (b, heads, T, head_dim) -> (b, T, d_out)
        context_vec = (attn_weights @ V).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

### 2.5 Transformer Block

Each block applies the **Pre-Norm** pattern with residual connections:

```
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

In [ ]:
class TransformerBlock(nn.Module):
    """One GPT-2 Transformer Block: Pre-LayerNorm + MHA + FFN + residuals."""

    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in           = cfg["emb_dim"],
            d_out          = cfg["emb_dim"],
            context_length = cfg["context_length"],
            num_heads      = cfg["n_heads"],
            dropout        = cfg["drop_rate"],
            qkv_bias       = cfg["qkv_bias"],
        )
        self.ff          = FeedForward(cfg)
        self.norm1       = LayerNorm(cfg["emb_dim"])
        self.norm2       = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # --- Attention sub-layer with residual ---
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # --- Feed-Forward sub-layer with residual ---
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

### 2.6 GPTModel — Putting It All Together

In [ ]:
class GPTModel(nn.Module):
    """Full GPT-2 model: token + positional embeddings → Transformer blocks → logits."""

    def __init__(self, cfg):
        super().__init__()
        self.tok_emb  = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb  = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)                                        # (B, T, C)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))   # (T, C)
        x = tok_embeds + pos_embeds   # broadcast over batch
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)       # (B, T, vocab_size)

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()   # disable dropout for inference
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

---
## Step 3: Tokenizer & Helper Utilities

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

def text_to_token_ids(text, tokenizer):
    """Encode text → 1-D integer tensor, then add a batch dimension."""
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)   # shape (1, T)

def token_ids_to_text(token_ids, tokenizer):
    """Decode a (1, T) or (T,) tensor of token IDs back to a string."""
    flat = token_ids.squeeze(0)                 # remove batch dim
    return tokenizer.decode(flat.tolist())

# Sanity check
sample_text = "Every effort moves you"
ids = text_to_token_ids(sample_text, tokenizer)
print("Token IDs:", ids)
print("Decoded  :", token_ids_to_text(ids, tokenizer))

---
## Step 4: Decoding Strategy 1 -- Greedy Decoding

<div class="alert alert-block alert-info">

**Greedy decoding** always selects the token with the **highest probability** at every step.

- **Pro:** Fast, deterministic, reproducible.
- **Con:** Repetitive, boring — the model gets stuck in loops.

</div>

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    """
    Greedy (argmax) token generation.

    Args:
        model         : GPTModel instance
        idx           : (batch, n_tokens) LongTensor of prompt token IDs
        max_new_tokens: number of new tokens to generate
        context_size  : maximum context the model accepts

    Returns:
        idx           : (batch, n_tokens + max_new_tokens) LongTensor
    """
    for _ in range(max_new_tokens):
        # Crop to the model's context window
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)          # (B, T, vocab_size)

        # Keep only the last token's logits → (B, vocab_size)
        logits = logits[:, -1, :]

        # Greedy: pick the token with the highest logit (= highest probability)
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)   # (B, 1)
        idx      = torch.cat((idx, idx_next), dim=1)             # (B, T+1)

    return idx

In [ ]:
start_context = "Every effort moves you"
token_ids = text_to_token_ids(start_context, tokenizer)

out_ids = generate_text_simple(
    model          = model,
    idx            = token_ids,
    max_new_tokens = 25,
    context_size   = GPT_CONFIG_124M["context_length"]
)
print("Greedy output:")
print(token_ids_to_text(out_ids, tokenizer))

---
## Step 5: Decoding Strategy 2 -- Temperature Scaling

<div class="alert alert-block alert-success">

Before applying softmax, we divide the logits by a **temperature** $T$:

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

| Temperature | Effect |
|---|---|
| $T < 1.0$ | **sharper** distribution — model is more confident, less creative |
| $T = 1.0$ | original model distribution |
| $T > 1.0$ | **flatter** distribution — model is more uncertain, more creative/random |
| $T \to 0$ | equivalent to greedy decoding |

</div>

In [ ]:
# Visualise how temperature reshapes a probability distribution
torch.manual_seed(0)
raw_logits = torch.tensor([2.0, 1.0, 0.5, 0.1, -0.5])
temps = [0.1, 0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(1, len(temps), figsize=(14, 3), sharey=True)
for ax, T in zip(axes, temps):
    probs = torch.softmax(raw_logits / T, dim=-1).numpy()
    ax.bar(range(len(probs)), probs, color="steelblue", edgecolor="white")
    ax.set_title(f"T = {T}")
    ax.set_xlabel("Token index")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
axes[0].set_ylabel("Probability")
fig.suptitle("Effect of Temperature on Softmax Distribution", fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 6: Decoding Strategy 3 — Top-k Sampling

<div class="alert alert-block alert-warning">

**Top-k sampling** restricts the next-token candidates to the $k$ tokens with the highest logits, then samples from that truncated distribution:

1. Find the $k$ largest logit values.
2. Set all other logits to $-\infty$ (zero probability after softmax).
3. Apply softmax and sample.

This prevents the model from ever picking a very unlikely token while still introducing controlled randomness.

</div>

In [ ]:
# Visualise top-k masking
torch.manual_seed(1)
sample_logits = torch.randn(10)   # 10-token vocabulary for clarity

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, k in zip(axes, [10, 5, 3]):
    logits_copy = sample_logits.clone()
    if k < len(logits_copy):
        kth_val = torch.topk(logits_copy, k).values[-1]   # smallest of top-k
        logits_copy[logits_copy < kth_val] = -float("inf")
    probs = torch.softmax(logits_copy, dim=-1).numpy()

    colors = ["steelblue" if p > 0 else "lightgray" for p in probs]
    ax.bar(range(len(probs)), probs, color=colors, edgecolor="white")
    ax.set_title(f"Top-k = {k}")
    ax.set_xlabel("Token index")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
axes[0].set_ylabel("Probability")
fig.suptitle("Top-k Masking — blue bars are kept, grey bars are zeroed", fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 7: Decoding Strategy 4 — Nucleus (Top-p) Sampling

<div class="alert alert-block alert-info">

**Top-p (nucleus) sampling** keeps the **smallest set** of top tokens whose cumulative probability exceeds $p$:

1. Sort tokens by descending probability.
2. Compute the cumulative sum.
3. Exclude all tokens whose *cumulative* probability mass exceeds $p$.
4. Renormalise and sample.

Unlike top-k, the nucleus size adapts: when the distribution is sharp, fewer tokens are kept; when flat, more are kept.

</div>

In [ ]:
def top_p_sampling(logits, p=0.9):
    """
    Apply nucleus (top-p) sampling to a 1-D logit tensor.

    Args:
        logits: 1-D tensor of shape (vocab_size,)
        p     : cumulative probability threshold (0 < p <= 1)

    Returns:
        sampled token index (scalar int)
    """
    probs = torch.softmax(logits, dim=-1)                       # probabilities

    # Sort descending
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Mask tokens beyond the nucleus
    # Shift cumulative by 1 so we always keep at least the top token
    sorted_indices_to_remove = (cumulative_probs - sorted_probs) > p
    sorted_probs[sorted_indices_to_remove] = 0.0
    sorted_probs /= sorted_probs.sum()   # renormalise

    # Sample and map back to original vocabulary index
    sampled_sorted_idx = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_indices[sampled_sorted_idx].item()


# Demo: show nucleus size at different p values
torch.manual_seed(42)
demo_logits = torch.randn(20)
for p in [0.5, 0.9, 0.95, 0.99]:
    probs = torch.softmax(demo_logits, dim=-1)
    sorted_probs, _ = torch.sort(probs, descending=True)
    cumsum = torch.cumsum(sorted_probs, dim=-1)
    nucleus_size = ((cumsum - sorted_probs) <= p).sum().item()
    print(f"  p = {p:.2f}  →  nucleus keeps {nucleus_size} / {len(demo_logits)} tokens")

---
## Step 8: Full `generate()` Function — Temperature + Top-k

<div class="alert alert-block alert-success">

The production-ready `generate()` function from Raschka's book combines:

- **Temperature scaling** for distribution sharpness control
- **Top-k sampling** to avoid catastrophically unlikely tokens
- **Multinomial sampling** (not argmax) for diversity
- **EOS token** support to stop generation early

</div>

In [ ]:
def generate(model, idx, max_new_tokens, context_size,
             temperature=1.0, top_k=None, eos_id=None):
    """
    Token generation with temperature scaling and optional top-k sampling.

    Args:
        model          : GPTModel (in eval mode)
        idx            : (1, T) LongTensor of prompt token IDs
        max_new_tokens : max tokens to generate
        context_size   : model's maximum context length
        temperature    : >1 = more random, <1 = more deterministic, 0 = greedy
        top_k          : if set, restrict to top-k tokens before sampling
        eos_id         : if set, stop when this token is generated

    Returns:
        idx: (1, T + generated_tokens) LongTensor
    """
    for _ in range(max_new_tokens):

        # ── 1. Crop context to model window ─────────────────────────────────
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)    # (B, T, vocab_size)

        # ── 2. Focus on the last time step ───────────────────────────────────
        logits = logits[:, -1, :]       # (B, vocab_size)

        # ── 3. Top-k filtering ───────────────────────────────────────────────
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]                     # k-th largest value
            logits  = torch.where(
                logits < min_val,
                torch.full_like(logits, -float("inf")),
                logits
            )

        # ── 4. Temperature scaling & sampling ────────────────────────────────
        if temperature > 0.0:
            logits   /= temperature
            probs     = torch.softmax(logits, dim=-1)       # (B, vocab_size)
            idx_next  = torch.multinomial(probs, num_samples=1)  # (B, 1)
        else:
            # temperature == 0 → greedy
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # ── 5. Early stopping on EOS token ───────────────────────────────────
        if eos_id is not None and (idx_next == eos_id).any():
            break

        # ── 6. Append generated token ────────────────────────────────────────
        idx = torch.cat((idx, idx_next), dim=1)   # (B, T+1)

    return idx

---
## Step 9: Comparing All Strategies Side-by-Side

<div class="alert alert-block alert-warning">
Note: The model has **random weights** (not pre-trained), so the text is nonsensical — but the decoding behaviour differences are clearly visible.
</div>

In [ ]:
torch.manual_seed(123)

prompt = "Every effort moves you"
token_ids = text_to_token_ids(prompt, tokenizer)
ctx = GPT_CONFIG_124M["context_length"]

configs = [
    {"label": "Greedy (T=0)",            "temperature": 0.0,  "top_k": None},
    {"label": "T=0.5  (sharp)",           "temperature": 0.5,  "top_k": None},
    {"label": "T=1.0  (base)",            "temperature": 1.0,  "top_k": None},
    {"label": "T=1.5  (creative)",        "temperature": 1.5,  "top_k": None},
    {"label": "Top-k=5,  T=1.0",          "temperature": 1.0,  "top_k": 5  },
    {"label": "Top-k=50, T=0.7",          "temperature": 0.7,  "top_k": 50 },
]

print(f"Prompt: \"{prompt}\"\n{'-'*60}")
for cfg in configs:
    torch.manual_seed(42)   # same seed → only strategy changes outcome
    out = generate(
        model          = model,
        idx            = token_ids.clone(),
        max_new_tokens = 20,
        context_size   = ctx,
        temperature    = cfg["temperature"],
        top_k          = cfg["top_k"],
    )
    text = token_ids_to_text(out, tokenizer)
    generated = text[len(prompt):]   # show only the new part
    print(f"[{cfg['label']:<22}]  {generated}")

---
## Step 10: EOS Token — Stopping Generation Early

<div class="alert alert-block alert-info">

In GPT-2 the **end-of-sequence** token (`<|endoftext|>`) has ID **50256**. We can pass `eos_id=50256` to `generate()` so that generation halts automatically as soon as the model produces it.

</div>

In [ ]:
eos_id = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]
print(f"<|endoftext|> token ID: {eos_id}")

torch.manual_seed(1)
out_ids = generate(
    model          = model,
    idx            = text_to_token_ids("The story ends", tokenizer),
    max_new_tokens = 50,
    context_size   = ctx,
    temperature    = 1.0,
    top_k          = 25,
    eos_id         = eos_id,
)
print("Generated:", token_ids_to_text(out_ids, tokenizer))
print(f"Total tokens generated: {out_ids.shape[1]}")

---
## Summary

| Strategy | Key idea | When to use |
|---|---|---|
| **Greedy** | `argmax` — pick highest probability | Factual tasks, benchmarks |
| **Temperature** | divide logits by $T$ before softmax | Always — tune to task |
| **Top-k** | zero out all but top-$k$ logits | Creative tasks, chat |
| **Top-p (nucleus)** | keep tokens up to cumulative prob $p$ | When $k$ should adapt dynamically |
| **Temperature + Top-k** | combine both | Most practical LLM deployments |

<div class="alert alert-block alert-success">

**Key takeaway:** The `generate()` function is where "intelligence" and "creativity" are balanced.  
Low temperature + small top-k → coherent but predictable.  
High temperature + large top-k → surprising but potentially incoherent.

</div>